#### **Predictive Fault Classification in UR3 Cobots Operation**

#### **Purpose:** : This project compares Random Forest and Logistic Regression classifiers and selects the best-performing model to predict and differentiate between healthy operation and faults (Robot Protective Stop) for a UR3 Cobot.

##### **Source:** UCI Machine Learning Repository - UR3 CobotOps Dataset.

##### **Author:** Bello Oluwatobi

##### **Last Updated:** December 8, 2025

### #1 Installing Libraries

In [ ]:
#  importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from imblearn.over_sampling import SMOTE

### #2 Loading the UR3Cobot dataset

In [ ]:
# loading the dataset from local storage
ur3_cobotops_data_file = "../data/dataset_02052023.xlsx"

df = pd.read_excel(ur3_cobotops_data_file)

### #3 Data Exploration

In [ ]:
# displaying the first 10 rows of the dataset
df.head(10)

In [ ]:
# displaying the shape of the dataset
df.shape

### #4 Data Preprocessing

In [ ]:
# checking for missing values in the dataset
df.isna().sum()

In [ ]:
# specifying feature columns and target column
feature_cols = [col for col in df.columns if 'Current' in col or 'Temperature' in col]

target_col = 'Robot_ProtectiveStop'

In [ ]:
# separating features and target variable
x = df[feature_cols]
y = df[target_col]

In [ ]:
# displaying the distribution of the target variable
healthy_count = sum(y == 0)
fault_count = sum(y == 1)
pct_fault = fault_count / (healthy_count + fault_count) * 100
pct_healthy = healthy_count / (healthy_count + fault_count) * 100
print(f"Prev Healthy Count: {pct_healthy:.2f}%")
print(f"Prev Fault Count: {pct_fault:.2f}%")

In [ ]:
# checking for missing values in the features dataset
x.isna().sum()

In [ ]:
# displaying the feature columns
x.columns

In [ ]:
# checking for missing values in the target variable dataset
y.isna().sum()

In [ ]:
# handling missing values by filling them with median values
x = x.fillna(x.median())

In [ ]:
# checking for missing values in the features dataset after filling empty values
x.isna().sum()

In [ ]:
# handling missing values in the target variable by filling them with mode value (due to O's and 1's)
y = y.fillna(y.mode().iloc[0])

In [ ]:
# checking for missing values in the target variable after filling empty values
y.isna().sum()

### #5 Splitting Data into Train and Test sets (80/20)

In [ ]:
# splitting the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, random_state=42, stratify=y
)

### #6 Data Transformation

In [ ]:
# feature scaling using StandardScaler
scaler = StandardScaler()
# fitting and transforming the training data
x_train_scaled = scaler.fit_transform(x_train)
# transforming the testing data to avoid data leakage
x_test_scaled = scaler.transform(x_test)

In [ ]:
# converting scaled arrays back to DataFrames for future use
x_train_final = pd.DataFrame(x_train_scaled, columns=feature_cols)
x_test_final = pd.DataFrame(x_test_scaled, columns=feature_cols)

### #7 Model Training

In [ ]:
# training the RandomForestClassifier model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_final, y_train)

In [ ]:
# training the LogisticRegression model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(x_train_final, y_train)

### #8 Model Evaluation

In [ ]:
# evaluating the RandomForestClassifier model
y_pred_rf = rf_model.predict(x_test_final)
print(classification_report(y_test, y_pred_rf, zero_division=0))

In [ ]:
# evaluating the LogisticRegression model
y_pred_lr = lr_model.predict(x_test_final)
print(classification_report(y_test, y_pred_lr, zero_division=0))

In [ ]:
# specifying the figure size and creating subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 1st plot: Random Forest Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, cmap='Blues', ax=ax1)
ax1.set_title("Confusion Matrix: Random Forest")

# 2nd plot: Logistic Regression Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, cmap='PuRd', ax=ax2)
ax2.set_title("Confusion Matrix: Logistic Regression")


plt.tight_layout()
plt.show()

### #9 Balancing the Dataset

In [ ]:
# applying Synthetic Minority Oversampling Technique (SMOTE) to handle class imbalance in the training data
smote = SMOTE(random_state=42)
x_resampled, y_resampled = smote.fit_resample(x_train_final, y_train)
# checking the count of fault instances before and after resampling
print(f"Prev Fault Count: {sum(y_train == 1)}")
print(f"Resampled Fault Count: {sum(y_resampled == 1)}")

### #10 Model Training based on the balanced dataset

In [ ]:
# training the RandomForestClassifier model on balanced data
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42)
rf_balanced.fit(x_resampled, y_resampled)

In [ ]:
# training the LogisticRegression model on balanced data
lr_balanced = LogisticRegression(max_iter=1000)
lr_balanced.fit(x_resampled, y_resampled)

### #11 Model Evaluation based on the balanced dataset

In [ ]:
# evaluating the balanced RandomForestClassifier model
y_pred_rf_balanced = rf_balanced.predict(x_test_final)
print(classification_report(y_test, y_pred_rf_balanced))

In [ ]:
# evaluating the balanced LogisticRegression model
y_pred_lr_balanced = lr_balanced.predict(x_test_final)
print(classification_report(y_test, y_pred_lr_balanced))

In [ ]:
# specifying the figure size and creating subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 1. First plot: The Confusion Matrix for the balanced Random Forest
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf_balanced, cmap='Blues', ax=ax1)
ax1.set_title("Confusion Matrix: Balanced Random Forest")

# 2. Second plot: The Confusion Matrix for the balanced Logistic Regression
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr_balanced, cmap='PuRd', ax=ax2)
ax2.set_title("Confusion Matrix: Balanced Logistic Regression")


plt.tight_layout()
plt.show()

In [ ]:
# selecting a sample from the test set for inference speed analysis
sample = x_test_final.iloc[0:1]

In [ ]:
# measuring inference time for balanced Random Forest model

# initializing list to store inference times
inf_times_rf = []

# starting timer
t0 = time.perf_counter()

# initializing for loop for 1000 iterations
for _ in range(1000):
  rf_balanced.predict(sample)
  rf_latency = ((time.perf_counter() - t0) / 1000) * 1000
  inf_times_rf.append(rf_latency)

# displaying average inference latency
print(f"Average Balanced Random Forest Inference Latency: {np.average(inf_times_rf):.4f} ms")

In [ ]:
# measuring inference time for balanced Logistic Regression model

# initializing list to store inference times
inf_times = []

# starting timer
t0 = time.perf_counter()

# initializing for loop for 1000 iterations
for _ in range(1000):
  lr_balanced.predict(sample)
  lr_latency = ((time.perf_counter() - t0) / 1000) * 1000
  inf_times.append(lr_latency)

# displaying average inference latency
print(f"Average Balanced Logistic Regression Inference Latency: {np.average(inf_times):.4f} ms")

In [ ]:
# measuring inference time for imbalanced Random Forest model

# initializing list to store inference times
inf_times = []

# starting timer
t0 = time.perf_counter()

# initializing for loop for 1000 iterations
for _ in range(1000):
  rf_model.predict(sample)
  rf_latency = ((time.perf_counter() - t0) / 1000) * 1000
  inf_times.append(rf_latency)

# displaying average inference latency
print(f"Average Normal/Imbalanced Random Forest Model Inference Latency: {np.average(inf_times):.4f} ms")

In [ ]:
# measuring inference time for imbalanced Logistic Regression model

# initializing list to store inference times
inf_times = []

# starting timer
t0 = time.perf_counter()

# initializing for loop for 1000 iterations
for _ in range(1000):
  lr_model.predict(sample)
  lr_latency = ((time.perf_counter() - t0) / 1000) * 1000
  inf_times.append(lr_latency)

# displaying average inference latency
print(f"Average Normal/Imbalanced Logistic Regression Model Inference Latency: {np.average(inf_times):.4f} ms")

In [ ]:
# performing feature importance analysis for the selected balanced Random Forest model
importances = pd.Series(rf_balanced.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='teal')
plt.title("Feature Importance for Fault Prediction")
plt.show()

### #12 Model Exporting for Deployment

In [ ]:
# exporting the balanced Random Forest model and scaler for Streamlit deployment
joblib.dump(rf_balanced, 'ur3_balanced_model.pkl')
joblib.dump(scaler, 'scaler.pkl')